### `tracemalloc` — stdlib, no install needed

> **Question it answers:** how many bytes did Python actually allocate for this object, including all external storage?

In [3]:
import tracemalloc

def allocated_bytes(factory):
    """Return bytes allocated by factory(), following all pointers."""
    tracemalloc.start()
    obj = factory()                              # run the thing to measure
    current, _ = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    tracemalloc.clear_traces()
    return obj, current                          # return obj so it's not GC'd before measure

In [4]:
_, s_bytes = allocated_bytes(set)
_, d_bytes = allocated_bytes(dict)
_, l_bytes = allocated_bytes(list)

print(f"empty set : {s_bytes:>6} bytes")
print(f"empty dict: {d_bytes:>6} bytes")   # now includes ma_keys
print(f"empty list: {l_bytes:>6} bytes")

empty set :    216 bytes
empty dict:     64 bytes
empty list:     56 bytes


Now compare as they grow:

In [7]:
sizes = [0, 10, 100, 1_000, 10_000, 100_000]

print(f"{'n':>8}  {'set':>10}  {'dict':>10}  {'list':>10}")
print("-" * 44)

for n in sizes:
    _, s = allocated_bytes(lambda n=n: set(range(n)))
    _, d = allocated_bytes(lambda n=n: {i: None for i in range(n)})
    _, l = allocated_bytes(lambda n=n: list(range(n)))
    print(f"{n:>8}  {s:>10,}  {d:>10,}  {l:>10,}")

       n         set        dict        list
--------------------------------------------
       0         216           0          56
      10         728         288         136
     100       8,408       4,624         856
    1000      56,760      60,664      31,832
   10000     836,280     607,126     391,832
  100000   7,386,731   8,434,672   3,991,832


### CPython's empty dict optimization

When you call `dict()`, CPython does **not** allocate a new keys object. It sets `ma_keys` to a global singleton called `Py_EMPTY_KEYS` that already existed at interpreter startup — before tracemalloc started watching. No new allocation happens, so tracemalloc correctly reports only the 64-byte object shell.

The first time you _add a key_, Python replaces that shared singleton with a freshly allocated `ma_keys`. Only then does tracemalloc see the real cost.

### Verify it with populated structures

In [5]:
import tracemalloc

def allocated_bytes(factory):
    tracemalloc.start()
    obj = factory()
    current, _ = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    tracemalloc.clear_traces()
    return obj, current

In [6]:
sizes = [0, 1, 8, 9, 100, 1_000, 10_000]

print(f"{'n':>6}  {'set':>10}  {'dict':>10}  {'list':>10}")
print("-" * 44)

for n in sizes:
    _, s = allocated_bytes(lambda n=n: set(range(n)))
    _, d = allocated_bytes(lambda n=n: {i: None for i in range(n)})
    _, l = allocated_bytes(lambda n=n: list(range(n)))
    print(f"{n:>6}  {s:>10,}  {d:>10,}  {l:>10,}")

     n         set        dict        list
--------------------------------------------
     0         216           0          56
     1         216         160          72
     8         728         288         120
     9         728         288         136
   100       8,408       4,624         856
  1000      56,760      60,664      31,832
 10000     836,280     607,126     391,832


You should see dict jump at `n=1` when the shared empty keys gets replaced. And at `n=9` you should see set jump — that's when `smalltable[8]` fills and set has to allocate an external table for the first time, losing its size advantage.

### The corrected mental model
```shell
dict()           → 64 bytes   (shell only — ma_keys is a shared global)
{"a": 1}         → 64 + ma_keys allocation  (first key triggers real allocation)

set()            → 216 bytes  (8-slot table always embedded inline, no sharing)
{1}              → still 216  (fits in the inline table, nothing new allocated)
set(range(9))    → 216 + external table  (overflows inline table)
```

So the empty set being "heavier" than empty dict is real — but it's partly because dict cheats with a shared singleton. Once both structures hold actual data, they converge.

---

### `memory_profiler` — install once, then Jupyter magic

> **Question it answers:** which _line_ in my function is the memory hotspot?

In Jupyter — one-time load per kernel:

In [2]:
%pip install memory-profiler

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install setuptools

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   ------------------------------- -------- 0.8/1.0 MB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 2.8 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
%load_ext memory_profiler

`%memit` — like `%timeit` but for memory:

In [2]:
%memit set(range(100_000))
%memit {i: None for i in range(100_000)}
%memit list(range(100_000))

peak memory: 84.34 MiB, increment: 3.82 MiB
peak memory: 84.41 MiB, increment: 0.07 MiB
peak memory: 85.17 MiB, increment: 0.76 MiB


`%mprun` — line-by-line, tells you where the memory went:

In [3]:
# Must define function in a file or %%writefile cell first — %mprun can't profile lambdas

def build_index(n: int) -> dict[str, list[int]]:
    keys  = [f"user:{i}" for i in range(n)]       # line A
    index = {k: [] for k in keys}                 # line B
    for i, k in enumerate(keys):
        index[k].append(i)                        # line C
    return index

%mprun -f build_index build_index(50_000)

ERROR: Could not find file C:\Users\Dennie\AppData\Local\Temp\ipykernel_22080\1983126667.py



Output shows incremental memory per line:

```shell
Line #    Mem usage    Increment   Line Contents
================================================
     1   65.1 MiB     65.1 MiB   def build_index(n):
     2   69.4 MiB      4.3 MiB       keys  = [f"user:{i}" ...]   ← strings are heavy
     3   75.1 MiB      5.7 MiB       index = {k: [] for k ...}   ← dict + empty lists
     4   75.1 MiB      0.0 MiB       for i, k in enumerate(keys):
     5   75.1 MiB      0.0 MiB           index[k].append(i)
```

---

### When to reach for which

|Tool|Use when|
|---|---|
|`tracemalloc`|Comparing exact byte costs, detecting leaks between two snapshots, inside tests|
|`%memit`|Quick sanity check: "is this approach meaningfully cheaper?"|
|`%mprun`|You have a function that uses too much memory and need to find the line|

`tracemalloc` is the surgeon's instrument — precise, programmatic, no install. `memory_profiler` is the flashlight — shows the shape of the problem fast.